## 04b Error Analysis of Selected Configuration

This notebook analyzes the disagreement cases of the final selected configuration from Notebook 04.

### Main goal

The purpose of this notebook is to explain why the selected model and threshold still make a small number of errors.

It answers questions such as:

- Which pairs are false positives and false negatives?
- Do the errors follow recurring patterns?
- What do these errors reveal about the strengths and limitations of semantic similarity for curriculum overlap detection?

### What this notebook does

- loads the final recommendation from Notebook 04
- loads the long-format pair similarity scores
- loads the labeled pair dataset with full review text
- rebuilds predictions for the selected configuration using the locked threshold
- separates disagreements into false positives and false negatives
- shows disagreement cases for both development and test
- supports qualitative interpretation of remaining errors

This notebook is a **qualitative follow-up** to Notebook 04.

Its purpose is not to improve the reported benchmark result after the fact, but to interpret the remaining disagreement cases of the final selected configuration more carefully.

In [1]:
# Cell 01 - Import libraries and define project paths

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

BENCHMARK_PAIRS_DIR = DATA_DIR / "benchmark_pairs"
THRESHOLD_TABLES_DIR = REPORTS_DIR / "tables" / "thresholds"
ERROR_ANALYSIS_TABLES_DIR = REPORTS_DIR / "tables" / "error_analysis"

PAIR_SCORES_CSV_PATH = BENCHMARK_PAIRS_DIR / "pair_similarity_scores_long.csv"
PAIR_SCORES_PARQUET_PATH = BENCHMARK_PAIRS_DIR / "pair_similarity_scores_long.parquet"
PAIRS_CSV_PATH = DATA_DIR / "gold_pairs_review_labeled.csv"
FINAL_RECOMMENDATION_CSV_PATH = THRESHOLD_TABLES_DIR / "final_threshold_recommendation.csv"

ERROR_ANALYSIS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Pair scores CSV:", PAIR_SCORES_CSV_PATH)
print("Pairs CSV:", PAIRS_CSV_PATH)
print("Final recommendation CSV:", FINAL_RECOMMENDATION_CSV_PATH)
print("Error analysis tables dir:", ERROR_ANALYSIS_TABLES_DIR)

Project root: C:\Users\user\Documents\thesis
Pair scores CSV: C:\Users\user\Documents\thesis\data\benchmark_pairs\pair_similarity_scores_long.csv
Pairs CSV: C:\Users\user\Documents\thesis\data\gold_pairs_review_labeled.csv
Final recommendation CSV: C:\Users\user\Documents\thesis\reports\tables\thresholds\final_threshold_recommendation.csv
Error analysis tables dir: C:\Users\user\Documents\thesis\reports\tables\error_analysis


In [2]:
# Cell 02 - Load Notebook 04 outputs and labeled pair data

if PAIR_SCORES_PARQUET_PATH.exists():
    pair_scores_long_df = pd.read_parquet(PAIR_SCORES_PARQUET_PATH)
else:
    pair_scores_long_df = pd.read_csv(PAIR_SCORES_CSV_PATH, encoding="utf-8")

pairs_df = pd.read_csv(PAIRS_CSV_PATH, encoding="utf-8", keep_default_na=False)
final_recommendation_df = pd.read_csv(FINAL_RECOMMENDATION_CSV_PATH, encoding="utf-8")

print("pair_scores_long_df shape:", pair_scores_long_df.shape)
print("pairs_df shape:", pairs_df.shape)
print("final_recommendation_df shape:", final_recommendation_df.shape)

display(final_recommendation_df)
display(pairs_df.head(2))

pair_scores_long_df shape: (23100, 15)
pairs_df shape: (1225, 16)
final_recommendation_df shape: (2, 17)


,recommendation_type,model_name,text_variant,similarity_metric,selection_rule,locked_threshold,shortlist_rank,average_precision,roc_auc,dev_f1,dev_f05,test_f1,test_f05,test_precision,test_recall,dev_predicted_positive_count,test_predicted_positive_count
0,primary_recommendation,openai_text-embedding-3-large,enriched,cosine,max_f1,0.673784,1,0.939595,0.994870,0.894737,0.923913,0.769231,0.892857,1.0,0.625,18,5
1,conservative_alternative,openai_text-embedding-3-large,main,cosine,max_f1,0.659979,2,0.927445,0.995913,0.871795,0.885417,0.769231,0.892857,1.0,0.625,19,5


,pair_id,institution_1,degree_programme_1,course_unit_code_1,course_name_1,review_text_1,institution_2,degree_programme_2,course_unit_code_2,course_name_2,review_text_2,overlap_label,review_status,review_notes,Unnamed: 14,Unnamed: 15
0,TE00BR86__TE00CN56,Turku AMK,Information and Communication Technology,TE00BR86,Back-End Development,Institution: Turku AMK\nCourse code: TE00BR86\nCourse name: Back-End Development\nObjective: After completing the course the student can: Knows the main alternatives technologies on the server-sid...,Turku AMK,Information and Communication Technology,TE00CN56,Internet Networks and Security,Institution: Turku AMK\nCourse code: TE00CN56\nCourse name: Internet Networks and Security\nObjective: After completing the course the student: can name the devices and services used to support co...,0,reviewed,,,
1,TE00BR86__TE00CS88,Turku AMK,Information and Communication Technology,TE00BR86,Back-End Development,Institution: Turku AMK\nCourse code: TE00BR86\nCourse name: Back-End Development\nObjective: After completing the course the student can: Knows the main alternatives technologies on the server-sid...,Turku AMK,Information and Communication Technology,TE00CS88,Introduction to Programming,Institution: Turku AMK\nCourse code: TE00CS88\nCourse name: Introduction to Programming\nObjective: After completing the course the student will be able to: Describe the program flow on a diagram....,0,reviewed,,,


In [3]:
# Cell 03 - Select the primary recommendation

primary_recommendation_df = final_recommendation_df[
    final_recommendation_df["recommendation_type"] == "primary_recommendation"
].copy()

if primary_recommendation_df.empty:
    raise ValueError("No row found with recommendation_type='primary_recommendation'.")

if len(primary_recommendation_df) != 1:
    raise ValueError(
        f"Expected exactly one primary_recommendation row, found {len(primary_recommendation_df)}."
    )

primary_row = primary_recommendation_df.iloc[0]

selected_model_name = primary_row["model_name"]
selected_text_variant = primary_row["text_variant"]
selected_similarity_metric = primary_row["similarity_metric"]
selected_selection_rule = primary_row["selection_rule"]
selected_threshold = float(primary_row["locked_threshold"])

print("Selected primary recommendation:")
print("  model_name       :", selected_model_name)
print("  text_variant     :", selected_text_variant)
print("  similarity_metric:", selected_similarity_metric)
print("  selection_rule   :", selected_selection_rule)
print("  locked_threshold :", f"{selected_threshold:.6f}")

Selected primary recommendation:
  model_name       : openai_text-embedding-3-large
  text_variant     : enriched
  similarity_metric: cosine
  selection_rule   : max_f1
  locked_threshold : 0.673784


In [4]:
# Cell 04 - Build evaluation table for the selected configuration

selected_scores_df = pair_scores_long_df[
    (pair_scores_long_df["model_name"] == selected_model_name) &
    (pair_scores_long_df["text_variant"] == selected_text_variant) &
    (pair_scores_long_df["similarity_metric"] == selected_similarity_metric)
].copy()

if selected_scores_df.empty:
    raise ValueError("No rows found for the selected configuration in pair_scores_long_df.")

selected_scores_df["ai_binary"] = (
    selected_scores_df["similarity_score"] >= selected_threshold
).astype(int)

selected_scores_df["human_binary"] = selected_scores_df["overlap_label_binary"].astype(int)

selected_scores_df["error_type"] = np.where(
    (selected_scores_df["human_binary"] == 0) & (selected_scores_df["ai_binary"] == 1),
    "false_positive",
    np.where(
        (selected_scores_df["human_binary"] == 1) & (selected_scores_df["ai_binary"] == 0),
        "false_negative",
        "correct"
    )
)

print("selected_scores_df shape:", selected_scores_df.shape)

display(
    selected_scores_df[[
        "pair_id",
        "split",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "error_type",
    ]].head(10)
)

selected_scores_df shape: (700, 18)


,pair_id,split,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,error_type
7700,TE00BR86__TE00CN56,dev,Back-End Development,Internet Networks and Security,0.460935,0,0,correct
7701,TE00BR86__TE00CS88,dev,Back-End Development,Introduction to Programming,0.501664,0,0,correct
7702,TE00BR86__TT00CD70,dev,Back-End Development,Data Networks,0.442427,0,0,correct
7703,TE00BR86__TT00CD71,dev,Back-End Development,Linux Basics,0.432349,0,0,correct
7704,TE00BR86__TT00CD72,dev,Back-End Development,Servers and containers,0.488781,0,0,correct
7705,TE00BR86__TT00CD75,dev,Back-End Development,Windows Basics,0.394785,0,0,correct
7706,TE00BR86__TT00CD77,dev,Back-End Development,Basics of Programming,0.506891,0,0,correct
7707,TE00BR86__TT00CD78,dev,Back-End Development,Frontend Programming,0.615639,0,0,correct
7708,TE00BR86__TT00CD79,dev,Back-End Development,Basics of Web Development,0.588990,0,0,correct
7709,TE00BR86__TT00CD80,dev,Back-End Development,JavaScript Programming,0.542693,0,0,correct


In [5]:
# Cell 05 - Attach full review texts and save the full disagreement table

pair_text_df = pairs_df[[
    "pair_id",
    "review_text_1",
    "review_text_2",
    "review_notes",
]].drop_duplicates(subset=["pair_id"]).copy()

error_analysis_df = selected_scores_df.merge(
    pair_text_df,
    on="pair_id",
    how="left",
    suffixes=("", "_from_pairs")
)

# Keep one clean review_notes column
if "review_notes_from_pairs" in error_analysis_df.columns:
    error_analysis_df["review_notes_final"] = error_analysis_df["review_notes_from_pairs"]
elif "review_notes" in error_analysis_df.columns:
    error_analysis_df["review_notes_final"] = error_analysis_df["review_notes"]
else:
    error_analysis_df["review_notes_final"] = ""

disagreement_df = error_analysis_df[
    error_analysis_df["human_binary"] != error_analysis_df["ai_binary"]
].copy()

DISAGREEMENT_TABLE_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "selected_configuration_disagreements.csv"
disagreement_df.to_csv(DISAGREEMENT_TABLE_CSV_PATH, index=False, encoding="utf-8")

print("Saved disagreement table to:", DISAGREEMENT_TABLE_CSV_PATH)
print("Total disagreements:", len(disagreement_df))
print("Columns available in disagreement_df:")
print(disagreement_df.columns.tolist())

display(
    disagreement_df[[
        "pair_id",
        "split",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "error_type",
        "review_notes_final",
    ]]
    .sort_values(["split", "error_type", "similarity_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

Saved disagreement table to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\selected_configuration_disagreements.csv
Total disagreements: 7
Columns available in disagreement_df:
['pair_id', 'course_unit_code_1', 'course_name_1', 'course_unit_code_2', 'course_name_2', 'review_notes', 'overlap_label_binary', 'split', 'model_name', 'model_type', 'text_variant', 'similarity_metric', 'similarity_score', 'embedding_dim', 'encoding_seconds', 'ai_binary', 'human_binary', 'error_type', 'review_text_1', 'review_text_2', 'review_notes_from_pairs', 'review_notes_final']


,pair_id,split,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,error_type,review_notes_final
0,TT00CE00__TT00CE05,dev,Data Analysis and Visualization,Data Analytics Project,0.660184,1,0,false_negative,"Both cover data analysis, visualization, and analytical reporting as core content."
1,TT00CD99__TT00CE00,dev,Data Sources and Data Preprocessing,Data Analysis and Visualization,0.642100,1,0,false_negative,Both explicitly cover data cleaning and preprocessing as core content.
2,TT00CD90__TT00CD91,dev,Software Design,Object Oriented Programming,0.561369,1,0,false_negative,"Both cover UML modeling, class/object design, and object-oriented software structure."
3,TT00CD71__TT00CD75,dev,Linux Basics,Windows Basics,0.687750,0,1,false_positive,
4,TT00CD87__TT00CE09,test,Information Security Technologies,Auditing and Penetration Testing,0.666506,1,0,false_negative,Both cover applied technical security methods for assessing and improving system security.
5,TT00CD88__TT00CE11,test,Hardening,Data Security Controls,0.665038,1,0,false_negative,Both cover applied technical controls used to harden and protect systems.
6,TT00CE14__TT00CE16,test,Malware Analysis,Reverse Engineering,0.636332,1,0,false_negative,"Both cover dynamic analysis, executable behavior investigation, low-level binary analysis tools."


In [6]:
# Cell 06 - Summarize disagreements by split and error type

disagreement_summary_df = (
    disagreement_df
    .groupby(["split", "error_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "error_type"])
    .reset_index(drop=True)
)

split_totals_df = (
    selected_scores_df
    .groupby("split")
    .agg(
        pair_count=("pair_id", "count"),
        human_positive_count=("human_binary", "sum"),
        ai_positive_count=("ai_binary", "sum"),
    )
    .reset_index()
)

print("Disagreement summary:")
display(disagreement_summary_df)

print("Split totals:")
display(split_totals_df)

Disagreement summary:


,split,error_type,count
0,dev,false_negative,3
1,dev,false_positive,1
2,test,false_negative,3


Split totals:


,split,pair_count,human_positive_count,ai_positive_count
0,dev,595,20,18
1,test,105,8,5


In [7]:
# Cell 07 - False positives only

false_positives_df = disagreement_df[
    disagreement_df["error_type"] == "false_positive"
].copy()

FALSE_POSITIVES_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "selected_configuration_false_positives.csv"
false_positives_df.to_csv(FALSE_POSITIVES_CSV_PATH, index=False, encoding="utf-8")

print("Saved false positives to:", FALSE_POSITIVES_CSV_PATH)
print("False positives:", len(false_positives_df))

display(
    false_positives_df[[
        "pair_id",
        "split",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "review_notes_final",
        "review_text_1",
        "review_text_2",
    ]]
    .sort_values(["split", "similarity_score"], ascending=[True, False])
    .reset_index(drop=True)
)

Saved false positives to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\selected_configuration_false_positives.csv
False positives: 1


,pair_id,split,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,review_notes_final,review_text_1,review_text_2
0,TT00CD71__TT00CD75,dev,Linux Basics,Windows Basics,0.68775,0,1,,"Institution: JAMK\nCourse code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS...",Institution: JAMK\nCourse code: TT00CD75\nCourse name: Windows Basics\nObjective: You understand and know how to use Windows management tools and resource monitoring tools. You are able to diagnos...


In [8]:
# Cell 08 - False negatives only

false_negatives_df = disagreement_df[
    disagreement_df["error_type"] == "false_negative"
].copy()

FALSE_NEGATIVES_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "selected_configuration_false_negatives.csv"
false_negatives_df.to_csv(FALSE_NEGATIVES_CSV_PATH, index=False, encoding="utf-8")

print("Saved false negatives to:", FALSE_NEGATIVES_CSV_PATH)
print("False negatives:", len(false_negatives_df))

display(
    false_negatives_df[[
        "pair_id",
        "split",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "human_binary",
        "ai_binary",
        "review_notes_final",
        "review_text_1",
        "review_text_2",
    ]]
    .sort_values(["split", "similarity_score"], ascending=[True, False])
    .reset_index(drop=True)
)

Saved false negatives to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\selected_configuration_false_negatives.csv
False negatives: 6


,pair_id,split,course_name_1,course_name_2,similarity_score,human_binary,ai_binary,review_notes_final,review_text_1,review_text_2
0,TT00CE00__TT00CE05,dev,Data Analysis and Visualization,Data Analytics Project,0.660184,1,0,"Both cover data analysis, visualization, and analytical reporting as core content.",Institution: JAMK\nCourse code: TT00CE00\nCourse name: Data Analysis and Visualization\nObjective: Data analysis and visualization are needed in solutions to both data-based and machine learning p...,"Institution: JAMK\nCourse code: TT00CE05\nCourse name: Data Analytics Project\nObjective: The project offers an opportunity to gain valuable experience in project management, data processing, deve..."
1,TT00CD99__TT00CE00,dev,Data Sources and Data Preprocessing,Data Analysis and Visualization,0.642100,1,0,Both explicitly cover data cleaning and preprocessing as core content.,Institution: JAMK\nCourse code: TT00CD99\nCourse name: Data Sources and Data Preprocessing\nObjective: Data preprocessing is an essential part of data analytics and machine learning projects. As p...,Institution: JAMK\nCourse code: TT00CE00\nCourse name: Data Analysis and Visualization\nObjective: Data analysis and visualization are needed in solutions to both data-based and machine learning p...
2,TT00CD90__TT00CD91,dev,Software Design,Object Oriented Programming,0.561369,1,0,"Both cover UML modeling, class/object design, and object-oriented software structure.","Institution: JAMK\nCourse code: TT00CD90\nCourse name: Software Design\nObjective: After completing the course, you will learn the principles, methods and tools of software engineering. The primar...",Institution: JAMK\nCourse code: TT00CD91\nCourse name: Object Oriented Programming\nObjective: You know how to design and document classes and objects and the interaction between them. You know ho...
3,TT00CD87__TT00CE09,test,Information Security Technologies,Auditing and Penetration Testing,0.666506,1,0,Both cover applied technical security methods for assessing and improving system security.,Institution: JAMK\nCourse code: TT00CD87\nCourse name: Information Security Technologies\nObjective: The student knows different information security techniques and is able to apply them. The stud...,"Institution: JAMK\nCourse code: TT00CE09\nCourse name: Auditing and Penetration Testing\nObjective: After completing the course, the student knows the general assessment and auditing criteria used..."
4,TT00CD88__TT00CE11,test,Hardening,Data Security Controls,0.665038,1,0,Both cover applied technical controls used to harden and protect systems.,Institution: JAMK\nCourse code: TT00CD88\nCourse name: Hardening\nObjective: The student knows and is able to apply different hardening methods. The student knows and understands the uses of diffe...,Institution: JAMK\nCourse code: TT00CE11\nCourse name: Data Security Controls\nObjective: The student learns to know the main categories of information security controls and to understand their im...
5,TT00CE14__TT00CE16,test,Malware Analysis,Reverse Engineering,0.636332,1,0,"Both cover dynamic analysis, executable behavior investigation, low-level binary analysis tools.","Institution: JAMK\nCourse code: TT00CE14\nCourse name: Malware Analysis\nObjective: The course focuses on the analysis of malware, emphasizing dynamic analysis methods and tools. Students will lea...","Institution: JAMK\nCourse code: TT00CE16\nCourse name: Reverse Engineering\nObjective: After completing the course, the student understands the basic concepts of reverse engineering and is able to..."


In [9]:
# Cell 09 - Compact quantitative error summary for the selected configuration

error_summary_rows = []

for split_name, split_df in selected_scores_df.groupby("split"):
    tp = int(((split_df["human_binary"] == 1) & (split_df["ai_binary"] == 1)).sum())
    tn = int(((split_df["human_binary"] == 0) & (split_df["ai_binary"] == 0)).sum())
    fp = int(((split_df["human_binary"] == 0) & (split_df["ai_binary"] == 1)).sum())
    fn = int(((split_df["human_binary"] == 1) & (split_df["ai_binary"] == 0)).sum())

    positive_total = tp + fn
    negative_total = tn + fp
    pair_total = len(split_df)

    error_summary_rows.append({
        "split": split_name,
        "pair_count": pair_total,
        "human_positive_count": positive_total,
        "human_negative_count": negative_total,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "positive_success_rate": tp / positive_total if positive_total else np.nan,
        "positive_error_rate": fn / positive_total if positive_total else np.nan,
        "negative_success_rate": tn / negative_total if negative_total else np.nan,
        "negative_error_rate": fp / negative_total if negative_total else np.nan,
    })

error_summary_df = pd.DataFrame(error_summary_rows).sort_values("split").reset_index(drop=True)

ERROR_SUMMARY_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "selected_configuration_error_summary.csv"
error_summary_df.to_csv(ERROR_SUMMARY_CSV_PATH, index=False, encoding="utf-8")

print("Saved error summary to:", ERROR_SUMMARY_CSV_PATH)
display(
    error_summary_df.style.format({
        "positive_success_rate": "{:.2%}",
        "positive_error_rate": "{:.2%}",
        "negative_success_rate": "{:.2%}",
        "negative_error_rate": "{:.2%}",
    })
)

Saved error summary to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\selected_configuration_error_summary.csv


,split,pair_count,human_positive_count,human_negative_count,tp,tn,fp,fn,positive_success_rate,positive_error_rate,negative_success_rate,negative_error_rate
0,dev,595,20,575,17,574,1,3,85.00%,15.00%,99.83%,0.17%
1,test,105,8,97,5,97,0,3,62.50%,37.50%,100.00%,0.00%


In [10]:
# Cell 10 - Categorize false negatives into simple semantic reason groups

def categorize_false_negative(row):
    text = " ".join([
        str(row["course_name_1"]).lower(),
        str(row["course_name_2"]).lower(),
        str(row["review_notes_final"]).lower(),
    ])

    if any(term in text for term in ["security", "hardening", "penetration", "audit", "control"]):
        return "security_overlap"
    if any(term in text for term in ["data analysis", "visualization", "preprocessing", "analytics", "data cleaning"]):
        return "data_analytics_overlap"
    if any(term in text for term in ["uml", "class", "object", "software design", "object-oriented"]):
        return "software_design_oop_overlap"
    if any(term in text for term in ["malware", "reverse engineering", "binary", "dynamic analysis"]):
        return "low_level_analysis_overlap"
    return "other_semantic_overlap"

false_negatives_df = false_negatives_df.copy()
false_negatives_df["reason_category"] = false_negatives_df.apply(categorize_false_negative, axis=1)

false_negative_reason_summary_df = (
    false_negatives_df["reason_category"]
    .value_counts()
    .rename_axis("reason_category")
    .reset_index(name="count")
)

FALSE_NEGATIVE_REASON_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "false_negative_reason_summary.csv"
false_negative_reason_summary_df.to_csv(FALSE_NEGATIVE_REASON_CSV_PATH, index=False, encoding="utf-8")

print("Saved false-negative reason summary to:", FALSE_NEGATIVE_REASON_CSV_PATH)
display(false_negative_reason_summary_df)

print("False negatives with assigned categories:")
display(
    false_negatives_df[[
        "pair_id",
        "split",
        "course_name_1",
        "course_name_2",
        "similarity_score",
        "reason_category",
        "review_notes_final",
    ]]
    .sort_values(["reason_category", "split", "similarity_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

Saved false-negative reason summary to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\false_negative_reason_summary.csv


,reason_category,count
0,security_overlap,2
1,data_analytics_overlap,2
2,software_design_oop_overlap,1
3,low_level_analysis_overlap,1


False negatives with assigned categories:


,pair_id,split,course_name_1,course_name_2,similarity_score,reason_category,review_notes_final
0,TT00CE00__TT00CE05,dev,Data Analysis and Visualization,Data Analytics Project,0.660184,data_analytics_overlap,"Both cover data analysis, visualization, and analytical reporting as core content."
1,TT00CD99__TT00CE00,dev,Data Sources and Data Preprocessing,Data Analysis and Visualization,0.642100,data_analytics_overlap,Both explicitly cover data cleaning and preprocessing as core content.
2,TT00CE14__TT00CE16,test,Malware Analysis,Reverse Engineering,0.636332,low_level_analysis_overlap,"Both cover dynamic analysis, executable behavior investigation, low-level binary analysis tools."
3,TT00CD87__TT00CE09,test,Information Security Technologies,Auditing and Penetration Testing,0.666506,security_overlap,Both cover applied technical security methods for assessing and improving system security.
4,TT00CD88__TT00CE11,test,Hardening,Data Security Controls,0.665038,security_overlap,Both cover applied technical controls used to harden and protect systems.
5,TT00CD90__TT00CD91,dev,Software Design,Object Oriented Programming,0.561369,software_design_oop_overlap,"Both cover UML modeling, class/object design, and object-oriented software structure."


## Error analysis interpretation

The selected primary configuration produced only a small number of disagreements against the expert labels.

### Main pattern

The error profile is clearly asymmetric:

- false positives are extremely rare
- false negatives are more common than false positives
- the selected operating point is therefore precision-oriented and moderately conservative

This pattern is consistent with the thresholding strategy used in Notebook 04. The chosen threshold avoids noisy overlap flags and keeps reviewer workload practical, but it also filters out some real overlaps with weaker or more distributed semantic similarity.

### Quantitative interpretation

On the development split, the selected configuration correctly identified **85.00%** of expert-labeled positive pairs and **99.83%** of expert-labeled negative pairs.

On the held-out test split, it correctly identified **62.50%** of positive pairs and **100.00%** of negative pairs.

These results indicate that the model is especially strong at avoiding incorrect overlap suggestions. Its main weakness is not false-positive noise, but missing some true overlaps.

### False positive interpretation

Only one false positive was observed:

- **Linux Basics** vs **Windows Basics**

This error is semantically understandable. Both courses describe operating-system concepts, system administration tasks, troubleshooting, and practical environment management. The model therefore captured a real thematic similarity, even though the expert judgment considered the overlap insufficient for a positive match.

This strengthens confidence in the model, because its only false positive is plausible rather than arbitrary.

### False negative interpretation

The false negatives are more informative than the false positives. They mainly occur in cases where overlap exists, but the overlap is:

- partial rather than near-identical
- expressed using different terminology
- embedded in broader course descriptions
- more applied or context-dependent than directly stated in the titles

The missed positive pairs cluster into a small number of interpretable categories. In the final error analysis, the most visible remaining pattern is **security-related overlap**, while the other misses also reflect broader thematic overlap in areas such as data analytics, software design, and low-level technical analysis.

This suggests that the selected threshold remains slightly conservative for moderate-overlap cases, especially when the shared content is practical, cross-cutting, or described indirectly.

### Overall conclusion

The selected primary configuration behaves credibly and usefully for a thesis-grade PoC:

- it keeps false positives extremely low
- it misses only a limited number of expert-labeled overlaps
- the missed overlaps are semantically understandable rather than random

Overall, the results suggest that the selected OpenAI enriched configuration is strong enough for the PoC, and that its remaining errors mainly reflect the difficulty of thresholding partial curriculum overlap in a small, imbalanced dataset.

In [11]:
# Cell 11 - Key findings summary table

primary_row = final_recommendation_df[
    final_recommendation_df["recommendation_type"] == "primary_recommendation"
].iloc[0]

dev_row = error_summary_df[error_summary_df["split"] == "dev"].iloc[0]
test_row = error_summary_df[error_summary_df["split"] == "test"].iloc[0]

false_positive_count = int((disagreement_df["error_type"] == "false_positive").sum())
false_negative_count = int((disagreement_df["error_type"] == "false_negative").sum())

top_reason_row = (
    false_negative_reason_summary_df.iloc[0]
    if "false_negative_reason_summary_df" in globals() and not false_negative_reason_summary_df.empty
    else None
)

key_findings_rows = [
    {
        "finding": "Primary recommended configuration",
        "value": f"{primary_row['model_name']} | {primary_row['text_variant']} | {primary_row['similarity_metric']} | {primary_row['selection_rule']}",
    },
    {
        "finding": "Locked threshold",
        "value": f"{primary_row['locked_threshold']:.6f}",
    },
    {
        "finding": "Development positive success rate",
        "value": f"{dev_row['positive_success_rate']:.2%}",
    },
    {
        "finding": "Development negative success rate",
        "value": f"{dev_row['negative_success_rate']:.2%}",
    },
    {
        "finding": "Test positive success rate",
        "value": f"{test_row['positive_success_rate']:.2%}",
    },
    {
        "finding": "Test negative success rate",
        "value": f"{test_row['negative_success_rate']:.2%}",
    },
    {
        "finding": "Total disagreement count",
        "value": str(len(disagreement_df)),
    },
    {
        "finding": "False positive count",
        "value": str(false_positive_count),
    },
    {
        "finding": "False negative count",
        "value": str(false_negative_count),
    },
    {
        "finding": "Most common false-negative reason",
        "value": (
            f"{top_reason_row['reason_category']} ({int(top_reason_row['count'])})"
            if top_reason_row is not None else "N/A"
        ),
    },
]

key_findings_df = pd.DataFrame(key_findings_rows)

KEY_FINDINGS_CSV_PATH = ERROR_ANALYSIS_TABLES_DIR / "error_analysis_key_findings_summary.csv"
key_findings_df.to_csv(KEY_FINDINGS_CSV_PATH, index=False, encoding="utf-8")

print("Saved key findings summary to:", KEY_FINDINGS_CSV_PATH)
display(key_findings_df)

Saved key findings summary to: C:\Users\user\Documents\thesis\reports\tables\error_analysis\error_analysis_key_findings_summary.csv


,finding,value
0,Primary recommended configuration,openai_text-embedding-3-large | enriched | cosine | max_f1
1,Locked threshold,0.673784
2,Development positive success rate,85.00%
3,Development negative success rate,99.83%
4,Test positive success rate,62.50%
5,Test negative success rate,100.00%
6,Total disagreement count,7
7,False positive count,1
8,False negative count,6
9,Most common false-negative reason,security_overlap (2)


## Final interpretation

The selected configuration produced only one false positive, which indicates strong precision-oriented behavior. Most remaining errors were false negatives, meaning the model was generally conservative rather than overly permissive.

The final results show that the model performs especially well at avoiding incorrect overlap suggestions, while still missing some genuine but less explicit overlaps. These missed cases appear to involve broader thematic similarity, practical overlap, or overlap expressed indirectly in the course descriptions rather than with closely matching wording.

This supports a practical thesis interpretation: the selected model-threshold combination is effective for surfacing strong and explicit curriculum overlaps while still missing some broader or more implicit overlaps. In a curriculum review workflow, this is often a safer failure mode than producing many false positives.

These findings do not weaken the selected configuration. Instead, they clarify its current performance boundary and provide useful direction for future improvements, such as richer course descriptions, additional metadata, or later experiments with domain-adapted models.